In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
np.random.seed(42) # For reproducibility

# Sensor range constants (example)
STRAIGHT   = (200, 320)   # finger fully extended
HALF       = (400, 560)   # finger half bent
BENT       = (680, 900)   # finger fully curled

# N.B:
# Accelerometer (g units, ESP32 typical range)
# Hand orientations affect which axis gravity falls on
# ACC roughly in g * 100 (raw ADC-like) — keeping as float g values here

# Gyroscope: static sign = near zero movement
GYRO_STATIC = (-0.15, 0.15)

In [3]:
# ASL Static Sign Definitions
# Each sign: (thumb, index, middle, ring, pinky, ax_center, ay_center, az_center)
# ax/ay/az in g units (gravity = ~1g on dominant axis for static pose)

signs = {
    # Letters (static ones --> all alphabets except J and Z)
    "A": dict(thumb=HALF,     index=BENT,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.9,  az=0.3),
    "B": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=STRAIGHT, pinky=STRAIGHT, ax=0.0,  ay=1.0,  az=0.1),
    "C": dict(thumb=HALF,     index=HALF,     middle=HALF,     ring=HALF,     pinky=HALF,     ax=0.7,  ay=0.5,  az=0.4),
    "D": dict(thumb=HALF,     index=STRAIGHT, middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.0,  ay=1.0,  az=0.2),
    "E": dict(thumb=HALF,     index=HALF,     middle=HALF,     ring=HALF,     pinky=HALF,     ax=0.1,  ay=0.8,  az=0.5),
    "F": dict(thumb=HALF,     index=HALF,     middle=STRAIGHT, ring=STRAIGHT, pinky=STRAIGHT, ax=0.2,  ay=0.9,  az=0.3),
    "G": dict(thumb=STRAIGHT, index=STRAIGHT, middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.9,  ay=0.2,  az=0.3),
    "H": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.9,  ay=0.1,  az=0.3),
    "I": dict(thumb=BENT,     index=BENT,     middle=BENT,     ring=BENT,     pinky=STRAIGHT, ax=0.0,  ay=1.0,  az=0.1),
    "K": dict(thumb=STRAIGHT, index=STRAIGHT, middle=HALF,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.9,  az=0.4),
    "L": dict(thumb=STRAIGHT, index=STRAIGHT, middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.3,  ay=0.9,  az=0.2),
    "M": dict(thumb=HALF,     index=BENT,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.5,  az=0.8),
    "N": dict(thumb=HALF,     index=BENT,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.6,  az=0.7),
    "O": dict(thumb=HALF,     index=HALF,     middle=HALF,     ring=HALF,     pinky=HALF,     ax=0.2,  ay=0.8,  az=0.5),
    "P": dict(thumb=STRAIGHT, index=STRAIGHT, middle=HALF,     ring=BENT,     pinky=BENT,     ax=0.5,  ay=0.5,  az=0.7),
    "Q": dict(thumb=STRAIGHT, index=STRAIGHT, middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.4,  ay=0.4,  az=0.8),
    "R": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.1,  ay=1.0,  az=0.1),
    "S": dict(thumb=HALF,     index=BENT,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.8,  az=0.5),
    "T": dict(thumb=HALF,     index=BENT,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.2,  ay=0.7,  az=0.6),
    "U": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.0,  ay=1.0,  az=0.1),
    "V": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.1,  ay=1.0,  az=0.2),
    "W": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=STRAIGHT, pinky=BENT,     ax=0.0,  ay=1.0,  az=0.1),
    "X": dict(thumb=BENT,     index=HALF,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.2,  ay=0.9,  az=0.3),
    "Y": dict(thumb=STRAIGHT, index=BENT,     middle=BENT,     ring=BENT,     pinky=STRAIGHT, ax=0.2,  ay=0.8,  az=0.5),

    # Numbers (static ones --> 1-9)
    "NUM_1": dict(thumb=BENT,     index=STRAIGHT, middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.0,  ay=1.0,  az=0.1),
    "NUM_2": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.0,  ay=1.0,  az=0.1),
    "NUM_3": dict(thumb=STRAIGHT, index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=BENT,     ax=0.1,  ay=1.0,  az=0.2),
    "NUM_4": dict(thumb=BENT,     index=STRAIGHT, middle=STRAIGHT, ring=STRAIGHT, pinky=STRAIGHT, ax=0.0,  ay=1.0,  az=0.1),
    "NUM_5": dict(thumb=STRAIGHT, index=STRAIGHT, middle=STRAIGHT, ring=STRAIGHT, pinky=STRAIGHT, ax=0.0,  ay=1.0,  az=0.1),
    "NUM_6": dict(thumb=STRAIGHT, index=STRAIGHT, middle=STRAIGHT, ring=BENT,     pinky=STRAIGHT, ax=0.1,  ay=1.0,  az=0.2),
    "NUM_7": dict(thumb=STRAIGHT, index=STRAIGHT, middle=STRAIGHT, ring=STRAIGHT, pinky=BENT,     ax=0.1,  ay=1.0,  az=0.2),
    "NUM_8": dict(thumb=STRAIGHT, index=STRAIGHT, middle=HALF,     ring=BENT,     pinky=BENT,     ax=0.1,  ay=0.9,  az=0.3),
    "NUM_9": dict(thumb=STRAIGHT, index=HALF,     middle=BENT,     ring=BENT,     pinky=BENT,     ax=0.2,  ay=0.9,  az=0.3),
}

In [4]:
# Noise levels
FLEX_NOISE  = 25    # ± ADC units
ACCEL_NOISE = 0.08  # ± g
GYRO_NOISE  = 0.05  # ± rad/s

def sample_flex(rng_tuple, n):
    center = (rng_tuple[0] + rng_tuple[1]) / 2
    return np.clip(
        np.random.normal(center, FLEX_NOISE, n),
        rng_tuple[0], rng_tuple[1]
    ).round(2)

def generate_sign_data(label, cfg, n_sessions=5, n_timesteps=20):
    rows = []
    for session in range(1, n_sessions + 1):
        session_id = f"{label}_S{session:02d}"
        for t in range(1, n_timesteps + 1):
            row = {
                "timestep":   t,
                "flex1":      sample_flex(cfg["thumb"],  1)[0],
                "flex2":      sample_flex(cfg["index"],  1)[0],
                "flex3":      sample_flex(cfg["middle"], 1)[0],
                "flex4":      sample_flex(cfg["ring"],   1)[0],
                "flex5":      sample_flex(cfg["pinky"],  1)[0],
                "ax": round(np.random.normal(cfg["ax"], ACCEL_NOISE), 4),
                "ay": round(np.random.normal(cfg["ay"], ACCEL_NOISE), 4),
                "az": round(np.random.normal(cfg["az"], ACCEL_NOISE), 4),
                "gx": round(np.random.uniform(*GYRO_STATIC), 4),
                "gy": round(np.random.uniform(*GYRO_STATIC), 4),
                "gz": round(np.random.uniform(*GYRO_STATIC), 4),
                "label":      label,
                "session_id": session_id,
            }
            rows.append(row)
    return rows

In [5]:
# Generating all signs
all_rows = []
for label, cfg in signs.items():
    all_rows.extend(generate_sign_data(label, cfg))

df = pd.DataFrame(all_rows, columns=[
    "timestep","flex1","flex2","flex3","flex4","flex5",
    "ax","ay","az","gx","gy","gz","label","session_id"
])

df.to_csv("my-own-asl", index=False)

In [ ]:
# Summary
print(f"Total rows    : {len(df)}")
print(f"Total signs   : {df['label'].nunique()}")
print(f"Signs covered : {sorted(df['label'].unique())}")
print(f"\nRows per sign :\n{df['label'].value_counts().sort_index()}")
print(f"\nSample rows:\n{df.head(5)}")
print(f"\nColumn dtypes:\n{df.dtypes}")

Total rows    : 3300
Total signs   : 33
Signs covered : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'NUM_1', 'NUM_2', 'NUM_3', 'NUM_4', 'NUM_5', 'NUM_6', 'NUM_7', 'NUM_8', 'NUM_9', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']

Rows per sign :
label
A        100
B        100
C        100
D        100
E        100
F        100
G        100
H        100
I        100
K        100
L        100
M        100
N        100
NUM_1    100
NUM_2    100
NUM_3    100
NUM_4    100
NUM_5    100
NUM_6    100
NUM_7    100
NUM_8    100
NUM_9    100
O        100
P        100
Q        100
R        100
S        100
T        100
U        100
V        100
W        100
X        100
Y        100
Name: count, dtype: int64

Sample rows:
   timestep   flex1   flex2   flex3   flex4   flex5      ax      ay      az  \
0         1  492.42  786.54  806.19  828.08  784.15  0.0813  1.0263  0.3614   
1         2  465.48  776.87  775.72  766.90  724.69  0.1760  0.9653  0.1781   
2         3  